# 01 — Veri hazırlama (her Colab oturumunda bir kez)
**CPU çalışma zamanı yeterli**; GPU kotası harcamaz. Secrets: `KAGGLE_API_TOKEN`
(repo private ise `GITHUB_TOKEN` da). Çıktı: `/content/dv_rgbt` ve `/content/meta`.
Her `%%time` hücresinin süresini ve son hücrenin disk çıktısını Claude'a getir (ADR 0004).

In [ ]:
import os
from google.colab import userdata
os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')
print('token yüklendi (değer yazdırılmadı)')

In [ ]:
# Repo private ise Secrets'a GITHUB_TOKEN ekle (fine-grained, sadece bu repo, 'Contents: read').
import os
from google.colab import userdata
try:
    tok = userdata.get('GITHUB_TOKEN')
except Exception:
    tok = None
url = f'https://{tok}@github.com/OgzhnTarhn/baykus.git' if tok else 'https://github.com/OgzhnTarhn/baykus.git'
if os.path.isdir('/content/baykus/.git'):
    !git -C /content/baykus pull -q {url} main
else:
    !git clone -q {url} /content/baykus
    !git -C /content/baykus remote set-url origin https://github.com/OgzhnTarhn/baykus.git
%cd /content/baykus
!git log --oneline -3

In [ ]:
%%time
!pip -q install uv
!uv pip install --system -q -e /content/baykus/python
!baykus-data --help | head -3

In [ ]:
%%time
!pip -q install kaggle
!kaggle datasets download brendanalvey/visdrone-dronevehicle -p /content/raw --unzip -q
!du -sh /content/raw

In [ ]:
# Ham kök: içinde train/ val/ test/ olan klasör (Kaggle zip'i bir alt klasör ekleyebilir)
from pathlib import Path
cands = [p for p in [Path('/content/raw'), *Path('/content/raw').glob('*')]
         if (p / 'train').is_dir() and (p / 'val').is_dir()]
assert cands, 'train/ ve val/ içeren klasör bulunamadı; /content/raw yapısına bak'
RAW = cands[0].as_posix()
print('RAW =', RAW)
!ls {RAW}/train

In [ ]:
%%time
!baykus-data verify --raw {RAW} --meta /content/meta
!baykus-data convert --raw {RAW} --meta /content/meta --out /content/dv_rgbt
!baykus-data illum --out /content/dv_rgbt --meta /content/meta --fig /content/figs
!baykus-data rgbt4 --out /content/dv_rgbt
!baykus-data coco --out /content/dv_rgbt --meta /content/meta

In [ ]:
!du -sh /content/dv_rgbt/* /content/raw
!df -h /content